## Data Cleaning and Standardization

### Objective

The purpose of this notebook is to improve the quality of the anonymized datasets by identifying and correcting common data quality issues.

The cleaning process focuses on:

- Standardizing column names
- Removing duplicate records
- Handling missing values
- Correcting inconsistent data types
- Validating assessment scores
- Preparing the datasets for loading into the project database

The cleaned datasets generated in this notebook will serve as the final source for the SQL database.

In [2]:
import pandas as pd
import numpy as np


In [3]:
attendance = pd.read_csv("../data/processed/term1/attendance.csv")

test1 = pd.read_csv("../data/processed/term1/test1.csv")

test2 = pd.read_csv("../data/processed/term1/test2.csv")

groupwork = pd.read_csv("../data/processed/term1/groupwork.csv")

projectwork = pd.read_csv("../data/processed/term1/projectwork.csv")

classscore = pd.read_csv("../data/processed/term1/classscore.csv")

examscore = pd.read_csv("../data/processed/term1/examscore.csv")

In [4]:
def cleaning_summary(df):

    summary = pd.DataFrame({
        "Column": df.columns,
        "Data Type": df.dtypes.values,
        "Missing Values": df.isna().sum().values,
        "Duplicate Values": df.duplicated().sum(),
        "Unique Values": df.nunique().values
    })

    return summary

In [5]:
cleaning_summary(test1)

,Column,Data Type,Missing Values,Duplicate Values,Unique Values
0,ENGLISH LANGUAGE,float64,44,42,8
1,SOCIAL STUDIES,float64,44,42,7
2,REL. & MORAL EDU.,float64,43,42,8
3,MATHEMATICS,float64,43,42,7
4,INTEGRATED SCIENCE,float64,44,42,14
5,COMPUTING,float64,44,42,10
6,FRENCH,float64,45,42,10
7,GH. LANG. (AKUAPEM TWI),float64,45,42,21
8,CAREER TECHNOLOGY,float64,45,42,4
9,CREATIVE ARTS & DESIGN,float64,44,42,13


### Removing Blank Rows

The exported Excel worksheets contain several blank rows at the end of the datasets. These rows do not represent student records but are interpreted as observations with missing values when imported into pandas.

Removing these rows improves data quality by reducing unnecessary missing values and ensuring that only valid student records remain for further processing.

In [6]:
# Function to remove blank rows
def remove_blank_rows(df):

    df = df.dropna(how="all")

    if "Student_ID" in df.columns:
        df = df.dropna(subset=["Student_ID"])

    return df.reset_index(drop=True)

In [7]:
# Applying to all Datasets in term 1
attendance = remove_blank_rows(attendance)

test1 = remove_blank_rows(test1)

test2 = remove_blank_rows(test2)

groupwork = remove_blank_rows(groupwork)

projectwork = remove_blank_rows(projectwork)

classscore = remove_blank_rows(classscore)

examscore = remove_blank_rows(examscore)

In [8]:
print("Attendance:", attendance.shape)
print("Test 1:", test1.shape)
print("Test 2:", test2.shape)
print("Group Work:", groupwork.shape)
print("Project Work:", projectwork.shape)
print("Class Score:", classscore.shape)
print("Exam Score:", examscore.shape)

Attendance: (59, 2)
Test 1: (60, 11)
Test 2: (60, 11)
Group Work: (59, 11)
Project Work: (59, 11)
Class Score: (59, 22)
Exam Score: (59, 22)


In [9]:
cleaning_summary(test1)

,Column,Data Type,Missing Values,Duplicate Values,Unique Values
0,ENGLISH LANGUAGE,float64,1,0,8
1,SOCIAL STUDIES,float64,1,0,7
2,REL. & MORAL EDU.,float64,0,0,8
3,MATHEMATICS,float64,0,0,7
4,INTEGRATED SCIENCE,float64,1,0,14
5,COMPUTING,float64,1,0,10
6,FRENCH,float64,2,0,10
7,GH. LANG. (AKUAPEM TWI),float64,2,0,21
8,CAREER TECHNOLOGY,float64,2,0,4
9,CREATIVE ARTS & DESIGN,float64,1,0,13


## Correcting Data Types

Some variables were imported using data types that are not appropriate for analysis. Attendance values were stored as floating-point numbers even though attendance is recorded as whole numbers.

The data types were corrected to improve consistency and prepare the datasets for loading into the SQL database.



One student record contained a missing attendance value. Rather than removing the student or assigning a value of zero, the missing attendance was replaced with the class average. This approach preserves the student record while minimizing the impact of the missing value on subsequent analyses.

In [13]:
attendance["Attendance"] = (
    attendance["Attendance"]
    .fillna(round(attendance["Attendance"].mean()))
    .astype(int)
)

In [14]:
attendance["Attendance"].isna().sum()

np.int64(0)